In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
from pathlib import Path


In [ ]:
# Conectar con la base de datos y cargar las 3 tablas
import sqlite3

# Crear la conexión a la base de datos
conn = sqlite3.connect('../datos/intermedios/analisis_inmobiliario.db')

listings = pd.read_sql_query("SELECT * FROM listings", conn)
listings_det = pd.read_sql_query("SELECT * FROM listings_det", conn)
precio = pd.read_sql_query("SELECT * FROM idealista", conn)

conn.close()

In [ ]:
listings.info()

In [ ]:
# Cambiar el tipo de dato de las columnas name, neighborhood, neighborhood_group, room_type a object
listings['name'] = listings['name'].astype(object)
listings['neighbourhood'] = listings['neighbourhood'].astype(object)
listings['neighbourhood_group'] = listings['neighbourhood_group'].astype(object)
listings['room_type'] = listings['room_type'].astype(object)

In [ ]:
listings_det.info()

In [ ]:
# Cambiar la columna description a object
listings_det['description'] = listings_det['description'].astype(object)

In [ ]:
precio.info()

In [ ]:
# Cambiar la columna distrito a object
precio['distrito'] = precio['distrito'].astype(object)

In [ ]:
# Merge inner join entre listings y listings_det
df_listings = listings.merge(listings_det, on='id', how='inner')

In [ ]:
df_listings.info()

In [ ]:
# Merge entre df_listings y precio (left join)
df = df_listings.merge(precio, left_on='neighbourhood_group', right_on='distrito', how='left')

In [ ]:
# Eliminar neighbourhood_group
df = df.drop(columns=['id','neighbourhood_group'])

In [ ]:
# Renombrar la columna price a precio_noche y precio a precio_m2
df = df.rename(columns={'price': 'precio_noche', 'precio': 'precio_m2'})

In [ ]:
'''# Eliminar filas con availability_365 menor a 180
df = df[df['availability_365'] >= 180]'''

# Darle el mismo valor de 365 días a availability_365
df['availability_365'] = 365

In [ ]:
df.shape

In [ ]:
df.head(5)

In [ ]:
def calcular_precio_noche_total(row):
    tasa_uso = 0.5
    if row['room_type'] == 'Entire home/apt':
        return row['precio_noche']
    elif row['room_type'] == 'Private room':
        if row['bedrooms'] > 1:
            return row['precio_noche'] * row['bedrooms'] * tasa_uso
        else:
            return row['precio_noche']
    elif row['room_type'] == 'Shared room':
        if row['accommodates'] > 1:
            return row['precio_noche'] * row['accommodates'] * tasa_uso
        else:
            return row['precio_noche']
    else:
        return np.nan

df['precio_noche_total'] = df.apply(calcular_precio_noche_total, axis=1)


In [ ]:
# Crear una variable ingreso anual que multiplique el precio_noche_total por estimated_occupancy_l365d
df['ingreso_anual'] = df['precio_noche_total'] * df['estimated_occupancy_l365d']

In [ ]:
df.head(5)

In [ ]:
condiciones_m2= [
    (df['bedrooms'] == 0),
    (df['bedrooms'] == 1),
    (df['bedrooms'] == 2),
    (df['bedrooms'] == 3) & (df['bathrooms'] < 2),
    (df['bedrooms'] == 3) & (df['bathrooms'] >= 2),
    (df['bedrooms'] > 3) | (df['bathrooms'] >= 3)
]

valores_m2 = [35, 50, 65, 90, 110, 140]

df['m2_estimado'] = np.select(condiciones_m2, valores_m2, default=np.nan)

In [ ]:
# Crear una nueva variable llamada coste_adquisicion que multiplique el m2_estimado por el precio_m2 y multiplicar por un ajuste de 0.75
df['coste_adquisicion'] = df['m2_estimado'] * df['precio_m2'] * 0.75

In [ ]:
df.head(15)

In [ ]:
# Crear una media de adquisicion por distrito y mostrarla ordenada de mayor a menor
df.groupby('distrito')['coste_adquisicion'].mean().reset_index().sort_values(by='coste_adquisicion', ascending=False)

In [ ]:
import sys
sys.path.append('../funciones')
from funciones import tourism_index

In [ ]:
# Cargar los CSV y convertirlos a diccionarios
df_pois = pd.read_csv('../datos/brutos/poi_madrid.csv')
pois = dict(zip(df_pois['nombre'], zip(df_pois['latitud'], df_pois['longitud'])))

df_weights = pd.read_csv('../datos/brutos/poi_madrid_weights.csv')
weights = dict(zip(df_weights['nombre'], df_weights['peso']))

# Calcular el atractivo turístico
df['atractivo_turistico'] = df.apply(
    lambda row: tourism_index(row['latitude'], row['longitude'], pois, weights), axis=1
)

In [ ]:
# Grafico de densidad de atractivo turistico
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df, x='atractivo_turistico', fill=True)
plt.title('Densidad de Atractivo Turístico')
plt.xlabel('Atractivo Turístico')
plt.ylabel('Densidad')
plt.show()

In [ ]:
# Ordenar df por atractivo turistico de menor a mayor
df.sort_values(by='atractivo_turistico', ascending=True)

In [ ]:
df.columns

In [ ]:
 # Genera un grafico de barras de bedrooms
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='accommodates', palette='viridis')
plt.title('Distribución de Accommodates')
plt.xlabel('Número de Accommodates')
plt.ylabel('Count')
plt.show()

In [ ]:
# Discretizar beds en las categorías '0', '1', '2', '3', '> 3'
df['beds_disc'] = pd.cut(df['beds'], bins=[-np.inf, 1, 2, 3, 4, np.inf], labels=['01_cama', '02_camas', '03_camas', '04_camas', '> 05_camas'])

In [ ]:
# Grafico de baaras de beds discretizado con tamaño personalizado
orden_beds = ['01_cama', '02_camas', '03_camas', '04_camas', '> 05_camas']
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='beds_disc', order=orden_beds, palette='viridis')
plt.title('Distribución de Beds Discretizado')
plt.xlabel('Número de Beds')
plt.ylabel('Count')
plt.show() 

In [ ]:
# Discretizar bathrooms en las categorías '<= 1', '1 - 2', '> 2'
df['bathrooms_disc'] = pd.cut(df['bathrooms'], bins=[-np.inf, 1, 2, np.inf], labels=['<= 1_toilet', '1 - 2 toilets', '> 2 toilets'])

In [ ]:
# Gráfico de barras de bathrooms discretizado con tamaño personalizado
orden = ['<= 1_toilet', '1 - 2 toilets', '> 2 toilets']
df['bathrooms_disc'] = pd.Categorical(df['bathrooms_disc'], categories=orden, ordered=True)
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='bathrooms_disc', palette='viridis', order=orden)
plt.title('Distribución de Bathrooms (discretizado)')
plt.xlabel('Número de Bathrooms')
plt.ylabel('Count')
plt.show()

In [ ]:
# Discretizar accommodates en las categorías '1', '2', '3', '4', '> 5'
df['accommodates_disc'] = pd.cut(df['accommodates'], bins=[-np.inf, 1, 2, 3, 4, np.inf], labels=['1', '2', '3', '4', '> 5'])

In [ ]:
# Grafico de barras de accommodates discretizado con tamaño personalizado
orden_accommodates = ['1', '2', '3', '4', '> 5']
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='accommodates_disc', order=orden_accommodates, palette='viridis')
plt.title('Distribución de Accommodates Discretizado')
plt.xlabel('Número de Accommodates')
plt.ylabel('Count')
plt.show()  
    

In [ ]:
# Discretizar beds en las categorías '<=1', '2', '3', '4', '> 5'
df['bedrooms_disc'] = pd.cut(df['beds'], bins=[-np.inf, 1, 2, 3, 4, np.inf], labels=['01_hab', '02_habs', '03_habs', '04_habs', '> 05_habs'])

In [ ]:
# Grafico de barras de bedrooms discretizado con tamaño personalizado
orden_bedrooms = ['01_hab', '02_habs', '03_habs', '04_habs', '> 05_habs']
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='bedrooms_disc', order=orden_bedrooms, palette='viridis')
plt.title('Distribución de Bedrooms Discretizado')
plt.xlabel('Número de Bedrooms')
plt.ylabel('Count')
plt.show()  

In [ ]:
# Crea una variable margen_bruto que sea el resultado de dividir el ingreso_anual entre el coste_adquisicion
df['margen_bruto'] = ((df['ingreso_anual'] / df['coste_adquisicion']) * 100).round(2)

In [ ]:
df.head(5)

In [ ]:
# Calcula el margen_bruto mediano por distrito
df.groupby('distrito')['margen_bruto'].median().reset_index().sort_values(by='margen_bruto', ascending=False)

In [ ]:
# Elimina el registro de neighbourhood 'Fuentelareina'
df = df[df['neighbourhood'] != 'Fuentelareina']

In [ ]:
# Crea una nueva tabla en la base de datos mercado inmobiliario.db que se llame tablon_analitico con el dataframe df
conn = sqlite3.connect('../datos/intermedios/analisis_inmobiliario.db')
df.to_sql('tablon_analitico', conn, if_exists='replace', index=False)
conn.close()